# NBA Quant AI v4 — GPU Deep Optimization (Colab)

**Purpose**: Deep Optuna hyperparameter search on the REAL feature engine (248+ features).

## Role Separation

| Component | Runs Where | Does What | Keep Running? |
|-----------|-----------|-----------|---------------|
| **HF Space S10** | HF Space 24/7 | Genetic exploration: 500 pop, 5 islands, try model types + feature combos | YES — do NOT stop |
| **This Notebook** | Colab (T4 GPU) | Deep Optuna (150 trials), GPU CatBoost/XGBoost, 4-model stacking | Run as needed |

**HF Space explores breadth** (which features, which model types).  
**Colab goes deep** (optimal hyperparams for the winning models, GPU speed).

## Instructions
1. Runtime → Change runtime type → **GPU (T4)**
2. Add Colab Secret: `DATABASE_URL` = your Supabase postgres URL
3. Run all cells

**Current best**: CatBoost Brier 0.2203 | Target: < 0.20

## 1. Setup — Clone Repo + Install

In [ ]:
import subprocess, sys, os

# Clone or pull the latest code
REPO_DIR = '/content/nomos-nba-agent'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/LBJLincoln/nomos-nba-agent.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'hf-space'))  # for features.engine import

# Install dependencies
_pkgs = [
    'psycopg2-binary', 'xgboost', 'lightgbm', 'catboost',
    'scikit-learn>=1.3', 'pandas', 'numpy', 'optuna', 'nba_api', 'geopy',
]
for pkg in _pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
print(f'PyTorch {torch.__version__} — CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)')

## 2. Load Secrets + Database

In [ ]:
# Load DATABASE_URL from Colab secrets
try:
    from google.colab import userdata
    os.environ.setdefault('DATABASE_URL', userdata.get('DATABASE_URL'))
    print('DATABASE_URL loaded from Colab secrets')
except Exception:
    print('Set DATABASE_URL manually if needed')

# Also try loading .env if present
for envf in ['.env', '.env.local']:
    if os.path.exists(envf):
        for line in open(envf):
            line = line.strip()
            if not line or line.startswith('#'): continue
            if line.startswith('export '): line = line[7:]
            if '=' in line:
                k, v = line.split('=', 1)
                os.environ.setdefault(k.strip(), v.strip().strip('\'"'))

## 3. Load Data + Build REAL Features (248+)

Uses the SAME `NBAFeatureEngine` as HF Space S10 — not a simplified version.

In [ ]:
import json, time, math, warnings
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss, accuracy_score, log_loss
from sklearn.model_selection import cross_val_predict
import xgboost as xgb
import lightgbm as lgbm
from catboost import CatBoostClassifier
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
(DATA_DIR / 'historical').mkdir(exist_ok=True)
(DATA_DIR / 'results').mkdir(exist_ok=True)

# ── Config ──
N_OPTUNA_TRIALS = 150    # Deep search with GPU
N_SPLITS = 5             # 5-fold walk-forward
PURGE_GAP = 5            # Temporal purge gap (same as HF Space)
RANDOM_STATE = 42

print(f'=== NBA QUANT AI v4 — GPU Deep Optimization ===')
print(f'Time: {datetime.now(timezone.utc).isoformat()}')
print(f'Optuna trials: {N_OPTUNA_TRIALS} per model')
print(f'CV splits: {N_SPLITS}')
print(f'GPU: {torch.cuda.is_available()}')

In [ ]:
# ── Pull NBA seasons (same as HF Space) ──
TEAM_MAP = {
    'Atlanta Hawks': 'ATL', 'Boston Celtics': 'BOS', 'Brooklyn Nets': 'BKN',
    'Charlotte Hornets': 'CHA', 'Chicago Bulls': 'CHI', 'Cleveland Cavaliers': 'CLE',
    'Dallas Mavericks': 'DAL', 'Denver Nuggets': 'DEN', 'Detroit Pistons': 'DET',
    'Golden State Warriors': 'GSW', 'Houston Rockets': 'HOU', 'Indiana Pacers': 'IND',
    'Los Angeles Clippers': 'LAC', 'Los Angeles Lakers': 'LAL', 'Memphis Grizzlies': 'MEM',
    'Miami Heat': 'MIA', 'Milwaukee Bucks': 'MIL', 'Minnesota Timberwolves': 'MIN',
    'New Orleans Pelicans': 'NOP', 'New York Knicks': 'NYK', 'Oklahoma City Thunder': 'OKC',
    'Orlando Magic': 'ORL', 'Philadelphia 76ers': 'PHI', 'Phoenix Suns': 'PHX',
    'Portland Trail Blazers': 'POR', 'Sacramento Kings': 'SAC', 'San Antonio Spurs': 'SAS',
    'Toronto Raptors': 'TOR', 'Utah Jazz': 'UTA', 'Washington Wizards': 'WAS',
}

def resolve(name):
    if name in TEAM_MAP: return TEAM_MAP[name]
    if len(name) == 3 and name.isupper(): return name
    for full, abbr in TEAM_MAP.items():
        if name in full: return abbr
    return name[:3].upper() if name else None

def pull_seasons():
    from nba_api.stats.endpoints import leaguegamefinder
    hist_dir = DATA_DIR / 'historical'
    existing = {f.stem.replace('games-', '') for f in hist_dir.glob('games-*.json')}
    targets = ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
    missing = [s for s in targets if s not in existing]
    if not missing:
        print(f'All {len(targets)} seasons cached')
        return
    for season in missing:
        print(f'Pulling {season}...')
        try:
            time.sleep(2)
            finder = leaguegamefinder.LeagueGameFinder(
                season_nullable=season, league_id_nullable='00',
                season_type_nullable='Regular Season', timeout=60
            )
            df = finder.get_data_frames()[0]
            if df.empty: continue
            pairs = {}
            for _, row in df.iterrows():
                gid = row['GAME_ID']
                if gid not in pairs: pairs[gid] = []
                pairs[gid].append({
                    'team_name': row.get('TEAM_NAME', ''),
                    'matchup': row.get('MATCHUP', ''),
                    'pts': int(row['PTS']) if row.get('PTS') is not None else None,
                    'game_date': row.get('GAME_DATE', ''),
                })
            games = []
            for gid, teams in pairs.items():
                if len(teams) != 2: continue
                home = next((t for t in teams if ' vs. ' in str(t.get('matchup', ''))), None)
                away = next((t for t in teams if ' @ ' in str(t.get('matchup', ''))), None)
                if not home or not away or home['pts'] is None: continue
                games.append({
                    'game_date': home['game_date'],
                    'home_team': home['team_name'], 'away_team': away['team_name'],
                    'home': {'team_name': home['team_name'], 'pts': home['pts']},
                    'away': {'team_name': away['team_name'], 'pts': away['pts']},
                })
            if games:
                (hist_dir / f'games-{season}.json').write_text(json.dumps(games))
                print(f'  {len(games)} games')
        except Exception as e:
            print(f'  Error: {e}')

def load_all_games():
    games = []
    for f in sorted((DATA_DIR / 'historical').glob('games-*.json')):
        data = json.loads(f.read_text())
        items = data if isinstance(data, list) else data.get('games', [])
        games.extend(items)
    games.sort(key=lambda g: g.get('game_date', g.get('date', '')))
    return games

# Pull data
pull_seasons()
games = load_all_games()
print(f'\nLoaded {len(games)} games')

In [ ]:
# ── Build features using the REAL NBAFeatureEngine ──
try:
    from features.engine import NBAFeatureEngine
    engine = NBAFeatureEngine()
    X, y, feature_names = engine.build(games)
    X = np.nan_to_num(np.array(X, dtype=np.float64))
    y = np.array(y, dtype=np.int32)
    print(f'NBAFeatureEngine: {X.shape[1]} features from {X.shape[0]} games')
    USE_REAL_ENGINE = True
except Exception as e:
    print(f'NBAFeatureEngine failed: {e}')
    print('Falling back to inline feature builder')
    USE_REAL_ENGINE = False
    # Fallback inline builder (same as app.py fallback)
    from hf_space_app_fallback import build_features_inline  # will define below if needed

print(f'X shape: {X.shape} | y shape: {y.shape}')
print(f'Home win rate: {y.mean():.3f}')

## 4. Genetic Feature Selection

Quick GA to select optimal ~100-150 features from the 248+ candidates.
This mirrors what HF Space does but runs faster on Colab.

In [ ]:
import random

def quick_evaluate(X_sub, y, n_splits=3):
    """Fast evaluation: XGBoost with walk-forward CV."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    model = xgb.XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, n_jobs=-1,
        tree_method='gpu_hist' if torch.cuda.is_available() else 'hist',
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )
    briers = []
    for ti, vi in tscv.split(X_sub):
        ti_safe = ti[:-PURGE_GAP] if len(ti) > PURGE_GAP + 50 else ti
        model.fit(X_sub[ti_safe], y[ti_safe])
        p = model.predict_proba(X_sub[vi])[:, 1]
        briers.append(brier_score_loss(y[vi], p))
    return np.mean(briers)

# Genetic feature selection
n_features = X.shape[1]
POP = 80
GENS = 60
TARGET = 100  # target ~100 features

# Initialize population
population = []
for _ in range(POP):
    prob = TARGET / n_features
    mask = [1 if random.random() < prob else 0 for _ in range(n_features)]
    population.append(mask)

best_brier = 1.0
best_mask = None
print(f'Starting GA: {POP} pop, {GENS} gens, {n_features} features, target ~{TARGET}')

for gen in range(GENS):
    # Evaluate
    scores = []
    for mask in population:
        selected = [i for i, v in enumerate(mask) if v]
        if len(selected) < 15 or len(selected) > 200:
            scores.append(0.30)
            continue
        X_sub = X[:, selected]
        scores.append(quick_evaluate(X_sub, y))

    # Track best
    gen_best_idx = np.argmin(scores)
    if scores[gen_best_idx] < best_brier:
        best_brier = scores[gen_best_idx]
        best_mask = population[gen_best_idx][:]
        n_sel = sum(best_mask)
        print(f'  Gen {gen:3d} | Brier {best_brier:.4f} | {n_sel} features')

    # Selection + crossover + mutation
    ranked = sorted(range(POP), key=lambda i: scores[i])
    new_pop = [population[ranked[0]][:], population[ranked[1]][:]]  # elitism

    while len(new_pop) < POP:
        # Tournament selection
        t1 = min(random.sample(range(POP), 5), key=lambda i: scores[i])
        t2 = min(random.sample(range(POP), 5), key=lambda i: scores[i])
        p1, p2 = population[t1], population[t2]

        # Two-point crossover
        if random.random() < 0.85:
            c1, c2 = sorted(random.sample(range(n_features), 2))
            child = p1[:c1] + p2[c1:c2] + p1[c2:]
        else:
            child = p1[:]

        # Mutation
        mut_rate = max(0.04, 0.12 * (0.997 ** gen))
        for i in range(n_features):
            if random.random() < mut_rate:
                child[i] = 1 - child[i]

        new_pop.append(child)

    population = new_pop

selected_features = [i for i, v in enumerate(best_mask) if v]
X_selected = X[:, selected_features]
print(f'\nGA complete: {len(selected_features)} features selected, Brier {best_brier:.4f}')

## 5. Deep Optuna Search (GPU Accelerated)

150 trials per model with GPU acceleration for XGBoost and CatBoost.

In [ ]:
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
optuna_results = {}
HAS_GPU = torch.cuda.is_available()

# ── XGBoost (GPU) ──
print(f'\nXGBoost Optuna ({N_OPTUNA_TRIALS} trials, GPU={HAS_GPU})...')
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
        'eval_metric': 'logloss', 'random_state': 42, 'n_jobs': -1,
    }
    if HAS_GPU:
        params['tree_method'] = 'gpu_hist'
        params['device'] = 'cuda'
    m = xgb.XGBClassifier(**params)
    briers = []
    for ti, vi in tscv.split(X_selected):
        ti_safe = ti[:-PURGE_GAP] if len(ti) > PURGE_GAP + 50 else ti
        m.fit(X_selected[ti_safe], y[ti_safe])
        briers.append(brier_score_loss(y[vi], m.predict_proba(X_selected[vi])[:, 1]))
    return np.mean(briers)

study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(xgb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
optuna_results['xgboost'] = {'brier': study_xgb.best_value, 'params': study_xgb.best_params}
print(f'  Best XGBoost: Brier={study_xgb.best_value:.4f}')

In [ ]:
# ── LightGBM ──
print(f'\nLightGBM Optuna ({N_OPTUNA_TRIALS} trials)...')
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
        'verbose': -1, 'random_state': 42, 'n_jobs': -1,
    }
    m = lgbm.LGBMClassifier(**params)
    briers = []
    for ti, vi in tscv.split(X_selected):
        ti_safe = ti[:-PURGE_GAP] if len(ti) > PURGE_GAP + 50 else ti
        m.fit(X_selected[ti_safe], y[ti_safe])
        briers.append(brier_score_loss(y[vi], m.predict_proba(X_selected[vi])[:, 1]))
    return np.mean(briers)

study_lgbm = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgbm.optimize(lgbm_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
optuna_results['lightgbm'] = {'brier': study_lgbm.best_value, 'params': study_lgbm.best_params}
print(f'  Best LightGBM: Brier={study_lgbm.best_value:.4f}')

In [ ]:
# ── CatBoost (GPU) ──
print(f'\nCatBoost Optuna ({N_OPTUNA_TRIALS} trials, GPU={HAS_GPU})...')
def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 800),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 5),
        'random_strength': trial.suggest_float('random_strength', 0, 5),
        'verbose': 0, 'random_state': 42,
    }
    if HAS_GPU:
        params['task_type'] = 'GPU'
        params['devices'] = '0'  # Single GPU — avoids "device already requested" error
    m = CatBoostClassifier(**params)
    briers = []
    for ti, vi in tscv.split(X_selected):
        ti_safe = ti[:-PURGE_GAP] if len(ti) > PURGE_GAP + 50 else ti
        m.fit(X_selected[ti_safe], y[ti_safe])
        briers.append(brier_score_loss(y[vi], m.predict_proba(X_selected[vi])[:, 1]))
    return np.mean(briers)

study_cat = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_cat.optimize(cat_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
optuna_results['catboost'] = {'brier': study_cat.best_value, 'params': study_cat.best_params}
print(f'  Best CatBoost: Brier={study_cat.best_value:.4f}')

## 6. Train All Models with Optimized Params

In [ ]:
print('\n=== Training all models with Optuna-optimized params ===')

# Build models with best Optuna params
xgb_params = {**optuna_results['xgboost']['params'], 'eval_metric': 'logloss', 'random_state': 42, 'n_jobs': -1}
if HAS_GPU:
    xgb_params['tree_method'] = 'gpu_hist'
    xgb_params['device'] = 'cuda'

lgbm_params = {**optuna_results['lightgbm']['params'], 'verbose': -1, 'random_state': 42, 'n_jobs': -1}

cat_params = {**optuna_results['catboost']['params'], 'verbose': 0, 'random_state': 42}
if HAS_GPU:
    cat_params['task_type'] = 'GPU'
    cat_params['devices'] = '0'

models = {
    'xgboost': xgb.XGBClassifier(**xgb_params),
    'lightgbm': lgbm.LGBMClassifier(**lgbm_params),
    'catboost': CatBoostClassifier(**cat_params),
    'random_forest': RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1),
    'extra_trees': ExtraTreesClassifier(n_estimators=500, max_depth=8, min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1),
    'logistic': LogisticRegression(max_iter=2000, C=1.0, random_state=42),
}

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

all_results = {}
for name, model in models.items():
    briers, accs = [], []
    for ti, vi in tscv.split(X_selected):
        ti_safe = ti[:-PURGE_GAP] if len(ti) > PURGE_GAP + 50 else ti
        Xtr = X_scaled[ti_safe] if 'logistic' in name else X_selected[ti_safe]
        Xva = X_scaled[vi] if 'logistic' in name else X_selected[vi]
        mc = type(model)(**model.get_params())
        mc.fit(Xtr, y[ti_safe])
        p = mc.predict_proba(Xva)[:, 1]
        briers.append(brier_score_loss(y[vi], p))
        accs.append(accuracy_score(y[vi], (p >= 0.5).astype(int)))
    all_results[name] = {'brier': round(np.mean(briers), 5), 'acc': round(np.mean(accs), 4)}
    print(f'  {name:20s} Brier={all_results[name]["brier"]:.4f} Acc={all_results[name]["acc"]:.3f}')

# Calibrated versions
print('\n=== Calibrated versions ===')
for name in list(models.keys()):
    try:
        briers = []
        for ti, vi in tscv.split(X_selected):
            ti_safe = ti[:-PURGE_GAP] if len(ti) > PURGE_GAP + 50 else ti
            Xtr = X_scaled[ti_safe] if 'logistic' in name else X_selected[ti_safe]
            Xva = X_scaled[vi] if 'logistic' in name else X_selected[vi]
            base = type(models[name])(**models[name].get_params())
            cal = CalibratedClassifierCV(base, method='sigmoid', cv=3)
            cal.fit(Xtr, y[ti_safe])
            briers.append(brier_score_loss(y[vi], cal.predict_proba(Xva)[:, 1]))
        cname = f'{name}_cal'
        all_results[cname] = {'brier': round(np.mean(briers), 5), 'acc': all_results[name]['acc']}
        delta = all_results[name]['brier'] - all_results[cname]['brier']
        print(f'  {cname:20s} Brier={all_results[cname]["brier"]:.4f} ({delta*10000:+.0f} pts)')
    except Exception as e:
        print(f'  {name}_cal failed: {e}')

## 7. 4-Model Stacking Ensemble (v4)

Same architecture as the fixed HF Space stacking:
- XGBoost + LightGBM + CatBoost + ExtraTrees base models
- Calibrated LogisticRegression meta-learner
- Temporal purging + inner 4-fold CV for OOF predictions

In [ ]:
print('\n=== Stacking v4: XGB + LGBM + CatBoost + ExtraTrees ===')

# CatBoost in stacking MUST use CPU to avoid CUDA device conflicts
cat_stack_params = {k: v for k, v in optuna_results['catboost']['params'].items()}
cat_stack_params['verbose'] = 0
cat_stack_params['random_state'] = 42
cat_stack_params['task_type'] = 'CPU'  # CRITICAL: avoid CUDA conflict in stacking
cat_stack_params['thread_count'] = -1

briers_stack = []
for fold, (ti, vi) in enumerate(tscv.split(X_selected)):
    ti_safe = ti[:-PURGE_GAP] if len(ti) > PURGE_GAP + 50 else ti
    X_tr, y_tr = X_selected[ti_safe], y[ti_safe]
    X_val, y_val = X_selected[vi], y[vi]

    # Build 4 base models
    base_models = [
        xgb.XGBClassifier(**{**optuna_results['xgboost']['params'],
                             'eval_metric': 'logloss', 'random_state': 42, 'n_jobs': -1,
                             'tree_method': 'hist'}),  # CPU for stacking inner CV
        lgbm.LGBMClassifier(**{**optuna_results['lightgbm']['params'],
                               'verbose': -1, 'random_state': 42, 'n_jobs': -1}),
        CatBoostClassifier(**cat_stack_params),
        ExtraTreesClassifier(n_estimators=300, max_depth=8, min_samples_leaf=5,
                            max_features='sqrt', random_state=42, n_jobs=-1),
    ]

    # OOF predictions from inner 4-fold CV
    inner_cv = TimeSeriesSplit(n_splits=4)
    oof_preds = np.zeros((len(X_tr), len(base_models)))
    for m_idx, bm in enumerate(base_models):
        try:
            oof = cross_val_predict(bm, X_tr, y_tr, cv=inner_cv, method='predict_proba')[:, 1]
            oof_preds[:, m_idx] = oof
        except Exception:
            oof_preds[:, m_idx] = 0.5

    # Calibrated meta-learner
    meta_base = LogisticRegression(C=0.5, max_iter=300, random_state=42)
    try:
        meta = CalibratedClassifierCV(meta_base, method='sigmoid', cv=3)
        meta.fit(oof_preds, y_tr)
    except Exception:
        meta = meta_base
        meta.fit(oof_preds, y_tr)

    # Train base models on full training fold
    val_preds = np.zeros((len(X_val), len(base_models)))
    for m_idx, bm in enumerate(base_models):
        try:
            bm_fresh = type(bm)(**bm.get_params())
            bm_fresh.fit(X_tr, y_tr)
            val_preds[:, m_idx] = bm_fresh.predict_proba(X_val)[:, 1]
        except Exception:
            val_preds[:, m_idx] = 0.5

    p = meta.predict_proba(val_preds)[:, 1]
    b = brier_score_loss(y_val, p)
    briers_stack.append(b)
    print(f'  Fold {fold}: Brier={b:.4f}')

stacking_brier = round(np.mean(briers_stack), 5)
all_results['stacking_v4'] = {'brier': stacking_brier, 'acc': 0}
print(f'\n  Stacking v4: Brier={stacking_brier:.4f}')

## 8. Results Summary + Save

In [ ]:
elapsed = time.time() if 'start_time' not in dir() else time.time()
best = min(all_results.items(), key=lambda x: x[1]['brier'])

print(f'\n{\'=\'*60}')
print(f'RESULTS')
print(f'{\'=\'*60}')
print(f'Games: {len(games)} | Raw features: {X.shape[1]} | Selected: {len(selected_features)}')
print(f'Genetic: {GENS} gens, pop {POP} | Optuna: {N_OPTUNA_TRIALS} trials x 3 models')
print(f'GPU: {HAS_GPU}')
print(f'\nBEST: {best[0]} — Brier {best[1]["brier"]:.4f}')
print(f'\nAll results:')
for name, r in sorted(all_results.items(), key=lambda x: x[1]['brier']):
    marker = ' <<<' if name == best[0] else ''
    print(f'  {name:25s} Brier={r["brier"]:.4f}{marker}')

# Save results
output = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'games': len(games),
    'raw_features': X.shape[1],
    'selected_features': len(selected_features),
    'selected_indices': selected_features,
    'ga_generations': GENS,
    'ga_population': POP,
    'optuna_trials_per_model': N_OPTUNA_TRIALS,
    'cv_splits': N_SPLITS,
    'gpu': HAS_GPU,
    'best_model': best[0],
    'best_brier': best[1]['brier'],
    'all_results': all_results,
    'optuna_params': optuna_results,
    'feature_names': feature_names[:20] if 'feature_names' in dir() else [],
}

out_file = DATA_DIR / 'results' / f'evolution-{datetime.now().strftime("%Y%m%d-%H%M")}.json'
out_file.write_text(json.dumps(output, indent=2, default=str))
print(f'\nSaved: {out_file}')

# Also save as latest
(DATA_DIR / 'results' / 'evolution-latest.json').write_text(json.dumps(output, indent=2, default=str))
print(f'Saved: data/results/evolution-latest.json')

## 9. Push Results to Supabase (Optional)

Logs the run so HF Space S10 can pick up the optimized params.

In [ ]:
DATABASE_URL = os.environ.get('DATABASE_URL', '')
if DATABASE_URL:
    try:
        import psycopg2
        conn = psycopg2.connect(DATABASE_URL)
        cur = conn.cursor()
        cur.execute('''
            INSERT INTO nba_evolution_runs (source, best_brier, best_model, features_selected, games, config, results)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        ''', (
            'colab_gpu_v4',
            best[1]['brier'],
            best[0],
            len(selected_features),
            len(games),
            json.dumps({'ga_gens': GENS, 'ga_pop': POP, 'optuna_trials': N_OPTUNA_TRIALS, 'gpu': HAS_GPU}),
            json.dumps(output, default=str),
        ))
        conn.commit()
        cur.close()
        conn.close()
        print('Results pushed to Supabase')
    except Exception as e:
        print(f'Supabase push failed: {e}')
else:
    print('No DATABASE_URL — skipping Supabase push')
    print('Add it in Colab Secrets (key icon, left sidebar) to enable.')